# Análise Exploratória de Dados — Churn de Clientes

Notebook organizado em duas partes:

1. **Tratamento da base** — visualização da base **bruta**, cada etapa de limpeza e a base **tratada**.
2. **Análise exploratória** — um gráfico por célula (gráficos 01 a 22).

**Antes de começar:** execute a célula de upload e envie o arquivo `base_churn_5000_VF2_aluno.xlsx`.

## 0. Setup e upload da base

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

ARQUIVO_ENTRADA = Path("base_churn_5000_VF2_aluno.xlsx")

# No Colab: abre o seletor de arquivos se a base ainda não foi enviada.
if not ARQUIVO_ENTRADA.exists():
    try:
        from google.colab import files
        print("Selecione o arquivo base_churn_5000_VF2_aluno.xlsx")
        files.upload()
    except ImportError:
        raise FileNotFoundError(f"Arquivo não encontrado: {ARQUIVO_ENTRADA.resolve()}")

print("Arquivo encontrado:", ARQUIVO_ENTRADA.exists())

## 1. Configurações do tratamento

In [ ]:
ARQUIVO_SAIDA = Path("base_churn_nova_tratada.xlsx")
DATA_CORTE = pd.Timestamp("2026-09-21")
DIAS_PREVISAO = 60
FIM_JANELA = DATA_CORTE + pd.Timedelta(days=DIAS_PREVISAO)

print("Data de corte:", DATA_CORTE.date())
print("Fim da janela de previsão:", FIM_JANELA.date())

## 2. Funções auxiliares

In [ ]:
def converter_data(valor):
    """Converte datas brasileiras e datas do Excel sem interromper o programa."""
    return pd.to_datetime(valor, dayfirst=True, errors="coerce")


def padronizar_contrato(valor):
    """Converte grafias diferentes para o domínio oficial de contratos."""
    if pd.isna(valor):
        return pd.NA
    valor = str(valor).strip().lower()
    mapa = {"mensal": "Mensal", "anual": "Anual", "bienal": "Bienal"}
    return mapa.get(valor, pd.NA)


def padronizar_pagamento(valor):
    """Converte grafias diferentes para o domínio oficial de pagamentos."""
    if pd.isna(valor):
        return pd.NA
    valor = str(valor).strip().lower()
    mapa = {"boleto": "Boleto", "cartão": "Cartão", "cartao": "Cartão", "pix": "Pix"}
    return mapa.get(valor, pd.NA)


def contar_chamados_90_dias(valor):
    """Conta chamados ocorridos nos 90 dias anteriores à data de corte."""
    if pd.isna(valor) or not str(valor).strip():
        return 0
    quantidade = 0
    inicio_janela = DATA_CORTE - pd.Timedelta(days=90)
    for texto_data in str(valor).split(","):
        data = converter_data(texto_data.strip())
        if pd.notna(data) and inicio_janela < data <= DATA_CORTE:
            quantidade += 1
    return quantidade


def meses_entre(data_inicio, data_fim):
    """Calcula o tempo completo de relacionamento em meses."""
    if pd.isna(data_inicio):
        return pd.NA
    meses = (data_fim.year - data_inicio.year) * 12
    meses += data_fim.month - data_inicio.month
    if data_fim.day < data_inicio.day:
        meses -= 1
    return max(0, meses)

## 3. Visualização da base **ANTES** do tratamento

### 3.1 Primeiras linhas e dimensões

In [ ]:
bruta = pd.read_excel(ARQUIVO_ENTRADA, dtype={"id_cliente": "string"})
bruta.columns = [col.strip() for col in bruta.columns]

print(f"Linhas: {bruta.shape[0]} | Colunas: {bruta.shape[1]}")
display(bruta.head(10))

### 3.2 Tipos de dados e valores nulos

In [ ]:
bruta.info()
print()
nulos_antes = bruta.isna().sum().rename("nulos").to_frame()
nulos_antes["% nulos"] = (nulos_antes["nulos"] / len(bruta) * 100).round(2)
display(nulos_antes)

### 3.3 Estatísticas descritivas (valores possivelmente inválidos)

In [ ]:
display(bruta.describe().T)

### 3.4 Categorias com grafias inconsistentes

In [ ]:
print("tipo_contrato:")
display(bruta["tipo_contrato"].value_counts(dropna=False).to_frame())
print("forma_pagamento:")
display(bruta["forma_pagamento"].value_counts(dropna=False).to_frame())

### 3.5 Duplicidades de cliente

In [ ]:
duplicados = bruta[bruta.duplicated("id_cliente", keep=False)].sort_values("id_cliente")
print("Linhas totais:", len(bruta))
print("Clientes distintos:", bruta["id_cliente"].nunique())
print("Linhas envolvidas em duplicidade:", len(duplicados))
display(duplicados.head(10))

## 4. Etapas do tratamento

### 4.1 Conversão de datas

In [ ]:
base = pd.read_excel(ARQUIVO_ENTRADA, dtype={"id_cliente": "string"})
base.columns = [col.strip() for col in base.columns]

base["data_aquisicao"] = base["data_aquisicao_cliente"].apply(converter_data)
base["data_churn"] = base["data_ocorrencia_churn"].apply(converter_data)

# Guarda os valores originais antes de qualquer correção.
base_original = base.copy()

display(base[["id_cliente", "data_aquisicao_cliente", "data_aquisicao",
              "data_ocorrencia_churn", "data_churn"]].head(10))

### 4.2 Padronização dos campos categóricos

In [ ]:
base["tipo_contrato_padronizado"] = base["tipo_contrato"].apply(padronizar_contrato)
base["forma_pagamento_padronizada"] = base["forma_pagamento"].apply(padronizar_pagamento)

base["contrato_ausente"] = base["tipo_contrato_padronizado"].isna().astype(int)
base["pagamento_ausente"] = base["forma_pagamento_padronizada"].isna().astype(int)
base["tipo_contrato_padronizado"] = base["tipo_contrato_padronizado"].fillna("Desconhecido")
base["forma_pagamento_padronizada"] = base["forma_pagamento_padronizada"].fillna("Desconhecido")

display(base["tipo_contrato_padronizado"].value_counts().to_frame())
display(base["forma_pagamento_padronizada"].value_counts().to_frame())

### 4.3 Validação e imputação das variáveis numéricas

In [ ]:
# Mensalidade válida: R$ 10,00 a R$ 500,00. Atrasos válidos: 0 a 6.
base["mensalidade_valida"] = base["mensalidade_rs"].where(base["mensalidade_rs"].between(10, 500))
base["mensalidade_ausente"] = base["mensalidade_valida"].isna().astype(int)

base["atrasos_validos"] = base["atrasos_6m"].where(base["atrasos_6m"].between(0, 6))
base["atrasos_ausente"] = base["atrasos_validos"].isna().astype(int)

# Mediana por tipo de contrato, com fallback para a mediana global.
mediana_mensalidade_global = base["mensalidade_valida"].median()
mediana_atrasos_global = base["atrasos_validos"].median()

mediana_mensalidade_contrato = base.groupby("tipo_contrato_padronizado")["mensalidade_valida"].transform("median")
mediana_mensalidade_contrato = mediana_mensalidade_contrato.fillna(mediana_mensalidade_global)

mediana_atrasos_contrato = base.groupby("tipo_contrato_padronizado")["atrasos_validos"].transform("median")
mediana_atrasos_contrato = mediana_atrasos_contrato.fillna(mediana_atrasos_global)

base["mensalidade_tratada"] = base["mensalidade_valida"].fillna(mediana_mensalidade_contrato).round(2)
base["atrasos_tratados"] = base["atrasos_validos"].fillna(mediana_atrasos_contrato).round().astype(int)

print("Mensalidades inválidas/ausentes corrigidas:", int(base["mensalidade_ausente"].sum()))
print("Atrasos inválidos/ausentes corrigidos:", int(base["atrasos_ausente"].sum()))
display(base[["mensalidade_rs", "mensalidade_tratada", "atrasos_6m", "atrasos_tratados"]].describe().T)

### 4.4 Remoção de duplicidades

In [ ]:
quantidade_antes = len(base)
base = base.drop_duplicates(subset=["id_cliente"], keep="first").copy()
quantidade_duplicidades = quantidade_antes - len(base)

print(f"Linhas antes: {quantidade_antes} | depois: {len(base)} | removidas: {quantidade_duplicidades}")

### 4.5 Seleção dos clientes ativos na data de corte

In [ ]:
clientes_ativos = base[
    (base["data_aquisicao"] <= DATA_CORTE)
    & (base["data_churn"].isna() | (base["data_churn"] > DATA_CORTE))
].copy()

print("Clientes ativos na data de corte:", len(clientes_ativos))

### 4.6 Tabela final de modelagem (`clientes`)

In [ ]:
clientes = pd.DataFrame()
clientes["id_cliente"] = clientes_ativos["id_cliente"]
clientes["data_corte"] = DATA_CORTE.date()
clientes["data_aquisicao_cliente"] = clientes_ativos["data_aquisicao"].dt.date
clientes["data_churn"] = clientes_ativos["data_churn"].dt.date
clientes["tempo_cliente_meses"] = clientes_ativos["data_aquisicao"].apply(
    lambda data: meses_entre(data, DATA_CORTE)
)
clientes["mensalidade_rs"] = clientes_ativos["mensalidade_tratada"]
clientes["mensalidade_ausente"] = clientes_ativos["mensalidade_ausente"]
clientes["chamados_suporte"] = clientes_ativos["data_chamado"].apply(contar_chamados_90_dias)
clientes["atrasos_6m"] = clientes_ativos["atrasos_tratados"]
clientes["atrasos_ausente"] = clientes_ativos["atrasos_ausente"]
clientes["tipo_contrato"] = clientes_ativos["tipo_contrato_padronizado"]
clientes["forma_pagamento"] = clientes_ativos["forma_pagamento_padronizada"]
clientes["contrato_ausente"] = clientes_ativos["contrato_ausente"]
clientes["pagamento_ausente"] = clientes_ativos["pagamento_ausente"]
clientes["fibra_optica"] = clientes_ativos["fibra_optica"].astype(int)
clientes["debito_automatico"] = clientes_ativos["debito_automatico"].astype(int)

# Churn = 1 se o cancelamento ocorrer nos próximos 60 dias.
clientes["churn"] = (
    clientes["data_churn"].notna()
    & (pd.to_datetime(clientes["data_churn"]) > DATA_CORTE)
    & (pd.to_datetime(clientes["data_churn"]) <= FIM_JANELA)
).astype(int)

clientes = clientes.reset_index(drop=True)
display(clientes.head(10))

### 4.7 Normalização dos chamados de suporte (`chamados`)

In [ ]:
chamados_lista = []
for _, linha in base.drop_duplicates("id_cliente").iterrows():
    if pd.isna(linha["data_chamado"]):
        continue
    for texto_data in str(linha["data_chamado"]).split(","):
        data = converter_data(texto_data.strip())
        if pd.notna(data):
            chamados_lista.append({"id_cliente": linha["id_cliente"], "data_chamado": data.date()})

chamados = pd.DataFrame(chamados_lista).drop_duplicates()
if not chamados.empty:
    chamados = chamados.sort_values(["id_cliente", "data_chamado"])

print("Total de chamados:", len(chamados))
display(chamados.head(10))

### 4.8 Auditoria das correções

In [ ]:
auditoria_linhas = []

for _, linha in base_original.iterrows():
    mensalidade_invalida = pd.isna(linha["mensalidade_rs"]) or not (10 <= linha["mensalidade_rs"] <= 500)
    if mensalidade_invalida:
        auditoria_linhas.append({
            "id_cliente": linha["id_cliente"], "campo": "mensalidade_rs",
            "problema": "Ausente ou fora do intervalo R$ 10–500",
            "tratamento": "Imputação pela mediana do tipo de contrato", "severidade": "Alta",
        })

    atrasos_invalidos = pd.isna(linha["atrasos_6m"]) or not (0 <= linha["atrasos_6m"] <= 6)
    if atrasos_invalidos:
        auditoria_linhas.append({
            "id_cliente": linha["id_cliente"], "campo": "atrasos_6m",
            "problema": "Ausente ou fora do intervalo 0–6",
            "tratamento": "Imputação pela mediana do tipo de contrato", "severidade": "Alta",
        })

    if pd.isna(linha["tipo_contrato"]) or str(linha["tipo_contrato"]).strip() != padronizar_contrato(linha["tipo_contrato"]):
        auditoria_linhas.append({
            "id_cliente": linha["id_cliente"], "campo": "tipo_contrato",
            "problema": "Valor não canônico ou ausente",
            "tratamento": "Padronização para o domínio oficial", "severidade": "Média",
        })

    if pd.isna(linha["forma_pagamento"]) or str(linha["forma_pagamento"]).strip() != padronizar_pagamento(linha["forma_pagamento"]):
        auditoria_linhas.append({
            "id_cliente": linha["id_cliente"], "campo": "forma_pagamento",
            "problema": "Valor não canônico ou ausente",
            "tratamento": "Padronização para o domínio oficial", "severidade": "Média",
        })

for cliente, quantidade in base_original["id_cliente"].value_counts().items():
    if quantidade > 1:
        auditoria_linhas.append({
            "id_cliente": cliente, "campo": "id_cliente",
            "problema": f"Cliente repetido {quantidade} vezes",
            "tratamento": "Retenção de uma ocorrência", "severidade": "Alta",
        })

auditoria = pd.DataFrame(auditoria_linhas)
display(auditoria.head(10))
display(auditoria.groupby(["campo", "severidade"]).size().rename("ocorrências").to_frame())

### 4.9 Exportação do Excel tratado

In [ ]:
dominios = pd.DataFrame(
    [
        ["tipo_contrato", "Mensal"], ["tipo_contrato", "Anual"],
        ["tipo_contrato", "Bienal"], ["tipo_contrato", "Desconhecido"],
        ["forma_pagamento", "Boleto"], ["forma_pagamento", "Cartão"],
        ["forma_pagamento", "Pix"], ["forma_pagamento", "Desconhecido"],
    ],
    columns=["campo", "valor_permitido"],
)

resumo = pd.DataFrame(
    [
        ["Arquivo original", ARQUIVO_ENTRADA.name],
        ["Data de corte", DATA_CORTE.date()],
        ["Fim da janela de previsão", FIM_JANELA.date()],
        ["Linhas originais", len(base_original)],
        ["Clientes distintos", base_original["id_cliente"].nunique()],
        ["Duplicidades removidas", quantidade_duplicidades],
        ["Clientes na base final", len(clientes)],
        ["Clientes com churn = 1", int(clientes["churn"].sum())],
        ["Clientes com churn = 0", int((clientes["churn"] == 0).sum())],
    ],
    columns=["indicador", "valor"],
)

with pd.ExcelWriter(ARQUIVO_SAIDA, engine="openpyxl", date_format="DD/MM/YYYY") as escritor:
    clientes.to_excel(escritor, index=False, sheet_name="clientes_modelagem")
    chamados.to_excel(escritor, index=False, sheet_name="chamados_suporte")
    auditoria.to_excel(escritor, index=False, sheet_name="auditoria")
    dominios.to_excel(escritor, index=False, sheet_name="dominios")
    resumo.to_excel(escritor, index=False, sheet_name="resumo")

# Formatação básica para facilitar a leitura no Excel.
workbook = load_workbook(ARQUIVO_SAIDA)
for planilha in workbook.worksheets:
    planilha.freeze_panes = "A2"
    planilha.auto_filter.ref = planilha.dimensions
    for celula in planilha[1]:
        celula.font = Font(bold=True, color="FFFFFF")
        celula.fill = PatternFill("solid", fgColor="1F4E78")
        celula.alignment = Alignment(horizontal="center")
    for coluna in range(1, planilha.max_column + 1):
        valores = [
            str(planilha.cell(linha, coluna).value or "")
            for linha in range(1, min(planilha.max_row, 200) + 1)
        ]
        largura = max(12, max(map(len, valores), default=12) + 2)
        planilha.column_dimensions[get_column_letter(coluna)].width = min(largura, 36)
workbook.save(ARQUIVO_SAIDA)

print("Base tratada criada com sucesso!")
display(resumo)

In [ ]:
# (Opcional) baixar o Excel tratado para o seu computador
try:
    from google.colab import files
    files.download(str(ARQUIVO_SAIDA))
except ImportError:
    print("Fora do Colab: arquivo salvo em", ARQUIVO_SAIDA.resolve())

## 5. Visualização da base **DEPOIS** do tratamento

### 5.1 Primeiras linhas e dimensões

In [ ]:
print(f"Linhas: {clientes.shape[0]} | Colunas: {clientes.shape[1]}")
display(clientes.head(10))

### 5.2 Tipos de dados e valores nulos

In [ ]:
clientes.info()
print()
nulos_depois = clientes.isna().sum().rename("nulos").to_frame()
nulos_depois["% nulos"] = (nulos_depois["nulos"] / len(clientes) * 100).round(2)
display(nulos_depois)

> Obs.: `data_churn` continua com nulos por definição — clientes que não cancelaram não têm data de cancelamento.

### 5.3 Estatísticas descritivas

In [ ]:
display(clientes.describe().T)

### 5.4 Categorias padronizadas

In [ ]:
display(clientes["tipo_contrato"].value_counts(dropna=False).to_frame())
display(clientes["forma_pagamento"].value_counts(dropna=False).to_frame())

### 5.5 Comparativo antes × depois (mensalidade e atrasos)

In [ ]:
comparativo = pd.DataFrame({
    "mensalidade_antes": bruta["mensalidade_rs"].describe(),
    "mensalidade_depois": clientes["mensalidade_rs"].describe(),
    "atrasos_antes": bruta["atrasos_6m"].describe(),
    "atrasos_depois": clientes["atrasos_6m"].describe(),
}).round(2)
display(comparativo)

# Parte 2 — Análise exploratória (gráficos)

In [ ]:
sns.set_theme(style="whitegrid", palette="deep")
CORES_CHURN = {0: "#4C78A8", 1: "#E45756"}
ROTULOS_CHURN = {0: "Permaneceu", 1: "Cancelou"}

# Lê a base tratada gerada na Parte 1.
base = pd.read_excel(ARQUIVO_SAIDA, sheet_name="clientes_modelagem")
base["status_churn"] = base["churn"].map(ROTULOS_CHURN)

ordem_classes = ["Permaneceu", "Cancelou"]
cores = {"Permaneceu": "#4C78A8", "Cancelou": "#E45756"}
ordem_contratos = ["Mensal", "Anual", "Bienal"]
base_sem_desconhecido = base[base["tipo_contrato"] != "Desconhecido"].copy()

variaveis = ["tempo_cliente_meses", "mensalidade_rs", "chamados_suporte", "atrasos_6m"]
rotulos = {
    "tempo_cliente_meses": "Tempo de cliente (meses)",
    "mensalidade_rs": "Mensalidade (R$)",
    "chamados_suporte": "Chamados de suporte",
    "atrasos_6m": "Atrasos nos últimos 6 meses",
}
print("Base para gráficos:", base.shape)

## 6. Gráficos iniciais

### 01 — Distribuição do churn

In [ ]:
contagem = base["status_churn"].value_counts().reindex(["Permaneceu", "Cancelou"])
fig, ax = plt.subplots(figsize=(8, 5))
barras = ax.bar(contagem.index, contagem.values, color=[CORES_CHURN[0], CORES_CHURN[1]])
ax.set_title("Distribuição do churn", fontsize=16, weight="bold")
ax.set_xlabel("Situação do cliente")
ax.set_ylabel("Quantidade de clientes")
for barra, valor in zip(barras, contagem.values):
    ax.text(barra.get_x() + barra.get_width()/2, valor + 35, f"{valor:,}".replace(",", "."), ha="center", weight="bold")
fig.tight_layout()
plt.show()

### 02 — Taxa de churn por tipo de contrato

In [ ]:
taxa_contrato = base.groupby("tipo_contrato", dropna=False)["churn"].agg(["mean", "count"]).reset_index()
taxa_contrato["taxa"] = taxa_contrato["mean"] * 100
taxa_contrato = taxa_contrato.sort_values("taxa", ascending=False)
fig, ax = plt.subplots(figsize=(8, 5))
barras = ax.bar(taxa_contrato["tipo_contrato"], taxa_contrato["taxa"], color="#F58518")
ax.set_title("Taxa de churn por tipo de contrato", fontsize=16, weight="bold")
ax.set_xlabel("Tipo de contrato")
ax.set_ylabel("Taxa de churn (%)")
ax.set_ylim(0, max(taxa_contrato["taxa"].max() * 1.2, 10))
for barra, taxa in zip(barras, taxa_contrato["taxa"]):
    ax.text(barra.get_x() + barra.get_width()/2, taxa + 0.7, f"{taxa:.1f}%", ha="center", weight="bold")
fig.tight_layout()
plt.show()

### 03 — Taxa de churn por forma de pagamento

In [ ]:
taxa_pagamento = base.groupby("forma_pagamento", dropna=False)["churn"].agg(["mean", "count"]).reset_index()
taxa_pagamento["taxa"] = taxa_pagamento["mean"] * 100
taxa_pagamento = taxa_pagamento.sort_values("taxa", ascending=False)
fig, ax = plt.subplots(figsize=(8, 5))
barras = ax.bar(taxa_pagamento["forma_pagamento"], taxa_pagamento["taxa"], color="#72B7B2")
ax.set_title("Taxa de churn por forma de pagamento", fontsize=16, weight="bold")
ax.set_xlabel("Forma de pagamento")
ax.set_ylabel("Taxa de churn (%)")
ax.set_ylim(0, max(taxa_pagamento["taxa"].max() * 1.2, 10))
for barra, taxa in zip(barras, taxa_pagamento["taxa"]):
    ax.text(barra.get_x() + barra.get_width()/2, taxa + 0.7, f"{taxa:.1f}%", ha="center", weight="bold")
fig.tight_layout()
plt.show()

### 04 — Distribuição da mensalidade por situação

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=base, x="status_churn", y="mensalidade_rs", order=ordem_classes,
            hue="status_churn", hue_order=ordem_classes, palette=cores, legend=False, ax=ax)
ax.set_title("Distribuição da mensalidade por situação", fontsize=16, weight="bold")
ax.set_xlabel("Situação do cliente")
ax.set_ylabel("Mensalidade (R$)")
fig.tight_layout()
plt.show()

### 05 — Taxa de churn por chamados de suporte

In [ ]:
chamados_taxa = base.groupby("chamados_suporte")["churn"].agg(["mean", "count"]).reset_index()
chamados_taxa["taxa"] = chamados_taxa["mean"] * 100
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(chamados_taxa["chamados_suporte"], chamados_taxa["taxa"], marker="o", color="#E45756", linewidth=2.5)
ax.set_title("Taxa de churn por chamados de suporte", fontsize=16, weight="bold")
ax.set_xlabel("Chamados nos 90 dias anteriores")
ax.set_ylabel("Taxa de churn (%)")
ax.set_xticks(sorted(chamados_taxa["chamados_suporte"].unique()))
fig.tight_layout()
plt.show()

### 06 — Taxa de churn por atrasos nos últimos 6 meses

In [ ]:
atrasos = base.groupby("atrasos_6m")["churn"].agg(["mean", "count"]).reset_index()
atrasos["taxa"] = atrasos["mean"] * 100
fig, ax = plt.subplots(figsize=(8, 5))
barras = ax.bar(atrasos["atrasos_6m"].astype(str), atrasos["taxa"], color="#B279A2")
ax.set_title("Taxa de churn por atrasos nos últimos 6 meses", fontsize=16, weight="bold")
ax.set_xlabel("Quantidade de atrasos")
ax.set_ylabel("Taxa de churn (%)")
ax.set_ylim(0, max(atrasos["taxa"].max() * 1.2, 10))
for barra, taxa in zip(barras, atrasos["taxa"]):
    ax.text(barra.get_x() + barra.get_width()/2, taxa + 0.7, f"{taxa:.1f}%", ha="center", weight="bold")
fig.tight_layout()
plt.show()

### 07 — Taxa de churn por tempo de relacionamento

In [ ]:
base["faixa_tempo"] = pd.cut(
    base["tempo_cliente_meses"],
    bins=[-1, 12, 24, 36, 60, 120],
    labels=["Até 1 ano", "1–2 anos", "2–3 anos", "3–5 anos", "Mais de 5 anos"],
)
tempo = base.groupby("faixa_tempo", observed=False)["churn"].agg(["mean", "count"]).reset_index()
tempo["taxa"] = tempo["mean"] * 100
fig, ax = plt.subplots(figsize=(9, 5))
barras = ax.bar(tempo["faixa_tempo"].astype(str), tempo["taxa"], color="#54A24B")
ax.set_title("Taxa de churn por tempo de relacionamento", fontsize=16, weight="bold")
ax.set_xlabel("Tempo como cliente")
ax.set_ylabel("Taxa de churn (%)")
ax.tick_params(axis="x", rotation=15)
ax.set_ylim(0, max(tempo["taxa"].max() * 1.2, 10))
for barra, taxa in zip(barras, tempo["taxa"]):
    ax.text(barra.get_x() + barra.get_width()/2, taxa + 0.7, f"{taxa:.1f}%", ha="center", weight="bold")
fig.tight_layout()
plt.show()

## 7. Gráficos adicionais

### 08 — Matriz de correlação

In [ ]:
colunas_numericas = ["tempo_cliente_meses", "mensalidade_rs", "chamados_suporte",
                     "atrasos_6m", "fibra_optica", "debito_automatico", "churn"]
correlacao = base[colunas_numericas].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(correlacao, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, ax=ax)
ax.set_title("Matriz de correlação das variáveis numéricas", fontsize=16, weight="bold", pad=15)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
fig.tight_layout()
plt.show()

### 09 — Proporção de churn por categoria e meio de pagamento

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, coluna, titulo, cor in [
    (axes[0], "tipo_contrato", "Proporção de churn por tipo de contrato", "#F58518"),
    (axes[1], "forma_pagamento", "Proporção de churn por meio de pagamento", "#72B7B2"),
]:
    base_grafico = base[base[coluna] != "Desconhecido"]
    proporcao = base_grafico.groupby(coluna, dropna=False)["churn"].mean().mul(100).sort_values(ascending=False)
    barras = ax.bar(proporcao.index.astype(str), proporcao.values, color=cor)
    ax.set_title(titulo, fontsize=13, weight="bold")
    ax.set_xlabel(coluna.replace("_", " ").title())
    ax.set_ylabel("Clientes com churn (%)")
    ax.set_ylim(0, max(proporcao.max() * 1.2, 10))
    ax.tick_params(axis="x", rotation=20)
    for barra, valor in zip(barras, proporcao.values):
        ax.text(barra.get_x() + barra.get_width() / 2, valor + 0.8, f"{valor:.1f}%", ha="center", weight="bold")
fig.suptitle("Proporção de churn por categoria e meio de pagamento", fontsize=17, weight="bold")
fig.tight_layout()
plt.show()

### 10 — Countplot: frequência de contratos por status de churn

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.countplot(
    data=base_sem_desconhecido, x="tipo_contrato", hue="status_churn",
    order=[x for x in ordem_contratos if x in base["tipo_contrato"].unique()],
    hue_order=ordem_classes, palette=cores, ax=ax,
)
ax.set_title("Frequência de contratos por status de churn", fontsize=16, weight="bold")
ax.set_xlabel("Tipo de contrato")
ax.set_ylabel("Quantidade de clientes")
ax.legend(title="Status do cliente")
fig.tight_layout()
plt.show()

### 11 — Heatmap: tipo de contrato × forma de pagamento

In [ ]:
ordem_linhas = [x for x in ["Mensal", "Anual", "Bienal"] if x in base["tipo_contrato"].unique()]
ordem_colunas = [x for x in ["Boleto", "Cartão", "Pix"] if x in base["forma_pagamento"].unique()]
taxa_churn = (
    base[base["tipo_contrato"].isin(ordem_linhas) & base["forma_pagamento"].isin(ordem_colunas)]
    .pivot_table(index="tipo_contrato", columns="forma_pagamento", values="churn", aggfunc="mean")
    .mul(100)
    .reindex(index=ordem_linhas, columns=ordem_colunas)
)
fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(taxa_churn, annot=True, fmt=".1f", cmap="YlOrRd", linewidths=0.5,
            vmin=0, vmax=max(50, float(taxa_churn.max().max()) * 1.05),
            cbar_kws={"label": "Taxa de churn (%)"}, ax=ax)
for texto in ax.texts:
    texto.set_text(f"{float(texto.get_text()):.1f}%")
ax.set_title("Taxa de churn: tipo de contrato × forma de pagamento", fontsize=16, weight="bold", pad=15)
ax.set_xlabel("Forma de pagamento")
ax.set_ylabel("Tipo de contrato")
fig.tight_layout()
plt.show()

### 12 — Histogramas das variáveis numéricas

In [ ]:
variaveis_hist = [
    ("tempo_cliente_meses", "Tempo de cliente (meses)"),
    ("mensalidade_rs", "Mensalidade (R$)"),
    ("chamados_suporte", "Chamados de suporte"),
    ("atrasos_6m", "Atrasos nos últimos 6 meses"),
]
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes = axes.flatten()
for ax, (coluna, titulo) in zip(axes, variaveis_hist):
    sns.histplot(base[coluna].dropna(), bins=20, kde=True, color="#4C78A8", ax=ax)
    ax.set_title(titulo, weight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("Frequência")
fig.suptitle("Histograma e distribuição global das variáveis numéricas", fontsize=17, weight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

## 8. Distribuição, KDE, violinos e escalas

### 13 — Box plots por classe de churn

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, coluna in zip(axes.flatten(), variaveis):
    sns.boxplot(data=base, x="status_churn", y=coluna, order=ordem_classes, hue="status_churn",
                hue_order=ordem_classes, palette=cores, legend=False, ax=ax)
    ax.set_title(rotulos[coluna], weight="bold")
    ax.set_xlabel("Status do cliente")
    ax.set_ylabel("")
fig.suptitle("Box plots das variáveis numéricas por classe de churn", fontsize=17, weight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

### 14 — Densidade (KDE) por classe de churn

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, coluna in zip(axes.flatten(), variaveis):
    for classe in ordem_classes:
        valores = base.loc[base["status_churn"] == classe, coluna].dropna()
        # KDE não é estável para um grupo com apenas um valor distinto.
        if valores.nunique() > 1:
            sns.kdeplot(valores, fill=True, alpha=0.25, linewidth=2, color=cores[classe], label=classe, ax=ax)
    ax.set_title(rotulos[coluna], weight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("Densidade")
    ax.legend(title="Classe")
fig.suptitle("Densidade de probabilidade (KDE) por classe de churn", fontsize=17, weight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

### 15 — Violin plots por classe de churn

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, coluna in zip(axes.flatten(), variaveis):
    sns.violinplot(data=base, x="status_churn", y=coluna, order=ordem_classes, hue="status_churn",
                   hue_order=ordem_classes, palette=cores, inner="quartile", cut=0, legend=False, ax=ax)
    ax.set_title(rotulos[coluna], weight="bold")
    ax.set_xlabel("Status do cliente")
    ax.set_ylabel("")
fig.suptitle("Violin plots das variáveis numéricas por classe de churn", fontsize=17, weight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

### 16 — Comparação de escalas: original × Min-Max × StandardScaler

In [ ]:
# Transformação apenas para visualização comparativa (fórmulas equivalentes às do scikit-learn).
X = base[variaveis].copy()
amplitudes = (X.max() - X.min()).replace(0, 1)
minmax = (X - X.min()) / amplitudes
desvios = X.std(ddof=0).replace(0, 1)
zscore = (X - X.mean()) / desvios

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, coluna in zip(axes.flatten(), variaveis):
    comparacao = pd.DataFrame({
        "Original": X[coluna],
        "Min-Max [0, 1]": minmax[coluna],
        "StandardScaler (Z-Score)": zscore[coluna],
    })
    comparacao_melt = comparacao.melt(var_name="escala", value_name="valor")
    sns.boxplot(data=comparacao_melt, x="escala", y="valor", color="#72B7B2", ax=ax)
    ax.set_title(rotulos[coluna], weight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("Valor transformado")
    ax.tick_params(axis="x", rotation=20)
fig.suptitle("Transformação de escala: original vs. Min-Max vs. StandardScaler", fontsize=17, weight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

### 16b — Transformação de escala da mensalidade (histogramas)

In [ ]:
variavel_aula = "mensalidade_rs"
valor_original = X[variavel_aula].dropna()
valor_minmax = minmax[variavel_aula].dropna()
valor_zscore = zscore[variavel_aula].dropna()

fig = plt.figure(figsize=(13, 7), facecolor="#F7F7F7")
grade = fig.add_gridspec(nrows=1, ncols=3, width_ratios=[1.0, 1.0, 1.0], wspace=0.28,
                         left=0.055, right=0.97, top=0.84, bottom=0.18)
ax_original = fig.add_subplot(grade[0, 0])
ax_minmax = fig.add_subplot(grade[0, 1])
ax_zscore = fig.add_subplot(grade[0, 2])

fig.suptitle("Transformação de Escala: Min-Max [0,1] vs. StandardScaler (Z-Score)",
             fontsize=18, weight="bold", color="#242A4A", x=0.50, y=0.95)

def histograma(ax, valores, bins, cor, titulo, xlabel):
    ax.hist(valores, bins=bins, color=cor, edgecolor="#263238", linewidth=0.65)
    ax.set_title(titulo, fontsize=10, weight="bold", color="#263238", pad=10)
    ax.set_xlabel(xlabel, fontsize=8)
    ax.set_ylabel("Frequência", fontsize=8)
    ax.tick_params(axis="both", labelsize=7)
    ax.grid(axis="y", alpha=0.25)
    ax.set_axisbelow(True)

histograma(ax_original, valor_original, 24, "#37474F", "1. Escala Original (R$)",
           f"[{valor_original.min():.2f}, {valor_original.max():.2f}]")
histograma(ax_minmax, valor_minmax, 24, "#28A887", "2. Normalização Min-Max", "Escala [0, 1]")
histograma(ax_zscore, valor_zscore, 24, "#E8872D", "3. Padronização StandardScaler", "Z-Score (Desvios Padrão)")

fig.text(0.055, 0.075, "ANÁLISE EXPLORATÓRIA • BASE DE CHURN • VARIÁVEL: MENSALIDADE", fontsize=8, color="#6B7280")
plt.show()

## 9. Análises bivariadas

### 17 — Tempo de cliente × chamados de suporte

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=base, x="tempo_cliente_meses", y="chamados_suporte", hue="status_churn",
                hue_order=ordem_classes, palette=cores, alpha=0.55, s=55, ax=ax)
ax.set_title("Análise bivariada: tempo de cliente × chamados de suporte", fontsize=16, weight="bold")
ax.set_xlabel("Tempo de cliente (meses)")
ax.set_ylabel("Chamados de suporte nos 90 dias anteriores")
ax.legend(title="Status do cliente")
fig.tight_layout()
plt.show()

### 18 — Countplot: frequência de contratos por status de churn

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.countplot(data=base_sem_desconhecido, x="tipo_contrato", hue="status_churn",
              order=ordem_contratos, hue_order=ordem_classes, palette=cores, ax=ax)
ax.set_title("Frequência de contratos por status de churn", fontsize=16, weight="bold")
ax.set_xlabel("Tipo de contrato")
ax.set_ylabel("Quantidade de clientes")
ax.legend(title="Status do cliente")
fig.tight_layout()
plt.show()

### 19 — FacetGrid: tempo × mensalidade por tipo de contrato

In [ ]:
grid = sns.lmplot(
    data=base_sem_desconhecido, x="tempo_cliente_meses", y="mensalidade_rs",
    col="tipo_contrato", col_order=ordem_contratos, hue="status_churn", hue_order=ordem_classes,
    palette=cores, height=4.8, aspect=0.95, scatter_kws={"alpha": 0.45, "s": 35}, fit_reg=False,
)
grid.set_axis_labels("Tempo de cliente (meses)", "Mensalidade (R$)")
grid.set_titles("Contrato: {col_name}")
grid.figure.suptitle("Tempo vs. mensalidade por tipo de contrato", y=1.05, fontsize=16, weight="bold")
plt.show()

### 20 — Regressão: tendência linear tempo × chamados

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=base, x="tempo_cliente_meses", y="chamados_suporte", hue="status_churn",
                hue_order=ordem_classes, palette=cores, alpha=0.35, s=35, ax=ax, legend=True)
for status in ordem_classes:
    dados_status = base[base["status_churn"] == status]
    sns.regplot(data=dados_status, x="tempo_cliente_meses", y="chamados_suporte", scatter=False,
                color=cores[status], line_kws={"linewidth": 2.5}, ax=ax)
ax.set_title("Tendência linear: tempo de cliente × chamados de suporte", fontsize=16, weight="bold")
ax.set_xlabel("Tempo de cliente (meses)")
ax.set_ylabel("Chamados de suporte")
ax.legend(title="Status do cliente")
fig.tight_layout()
plt.show()

### 21 — Strip plot: mensalidade por chamados e status de churn

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
sns.stripplot(data=base, x="chamados_suporte", y="mensalidade_rs", hue="status_churn",
              hue_order=ordem_classes, palette=cores, dodge=True, jitter=0.22, alpha=0.5, size=4, ax=ax)
ax.set_title("Mensalidade por número de chamados e status de churn", fontsize=16, weight="bold")
ax.set_xlabel("Chamados de suporte")
ax.set_ylabel("Mensalidade (R$)")
ax.legend(title="Status do cliente")
fig.tight_layout()
plt.show()

### 22 — Letter-value plot: tempo de contrato por status de churn

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
sns.boxenplot(data=base, x="status_churn", y="tempo_cliente_meses", order=ordem_classes,
              hue="status_churn", hue_order=ordem_classes, palette=cores, legend=False, ax=ax)
ax.set_title("Letter-value plot: tempo de contrato por status de churn", fontsize=16, weight="bold")
ax.set_xlabel("Status do cliente")
ax.set_ylabel("Tempo de contrato (meses)")
fig.tight_layout()
plt.show()